In [ ]:
"""Reusable model components for multi-frame OCR."""
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import (
    EfficientNet_B0_Weights,
    ResNet34_Weights,
    efficientnet_b0,
    resnet34,
)


# ════════════════════════════════════════════════════════════════
# TPS-STN
# Source: ClovaAI deep-text-recognition-benchmark (ICCV 2019)
# ════════════════════════════════════════════════════════════════

class TPSLocalizationNetwork(nn.Module):
    def __init__(self, F: int, I_channel_num: int = 3):
        super().__init__()
        self.F = F
        self.conv = nn.Sequential(
            nn.Conv2d(I_channel_num, 64,  3, 1, 1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64,  128, 3, 1, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, 1, 1, bias=False), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.localization_fc1 = nn.Sequential(nn.Linear(512, 256), nn.ReLU(True))
        self.localization_fc2 = nn.Linear(256, F * 2)
        self.localization_fc2.weight.data.fill_(0)
        ctrl_pts_x        = np.linspace(-1.0, 1.0, int(F / 2))
        ctrl_pts_y_top    = np.linspace(0.0, -1.0, num=int(F / 2))
        ctrl_pts_y_bottom = np.linspace(1.0,  0.0, num=int(F / 2))
        initial_bias = np.concatenate([
            np.stack([ctrl_pts_x, ctrl_pts_y_top],    axis=1),
            np.stack([ctrl_pts_x, ctrl_pts_y_bottom], axis=1),
        ], axis=0)
        self.localization_fc2.bias.data = torch.from_numpy(initial_bias).float().view(-1)

    def forward(self, batch_I):
        B = batch_I.size(0)
        return self.localization_fc2(self.localization_fc1(self.conv(batch_I).view(B, -1))).view(B, self.F, 2)


class TPSGridGenerator(nn.Module):
    def __init__(self, F: int, I_r_size: tuple):
        super().__init__()
        self.eps = 1e-6
        self.I_r_height, self.I_r_width = I_r_size
        self.F = F
        self.C = self._build_C(F)
        P = self._build_P(self.I_r_width, self.I_r_height)
        self.register_buffer('inv_delta_C', torch.tensor(self._build_inv_delta_C(F, self.C)).float())
        self.register_buffer('P_hat',       torch.tensor(self._build_P_hat(F, self.C, P)).float())

    def _build_C(self, F):
        ctrl_pts_x = np.linspace(-1.0, 1.0, int(F / 2))
        return np.concatenate([
            np.stack([ctrl_pts_x, -1 * np.ones(int(F / 2))], axis=1),
            np.stack([ctrl_pts_x,      np.ones(int(F / 2))], axis=1),
        ], axis=0)

    def _build_inv_delta_C(self, F, C):
        hat_C = np.zeros((F, F), dtype=float)
        for i in range(F):
            for j in range(i, F):
                r = np.linalg.norm(C[i] - C[j])
                hat_C[i, j] = hat_C[j, i] = r
        np.fill_diagonal(hat_C, 1)
        hat_C = (hat_C ** 2) * np.log(hat_C)
        delta_C = np.concatenate([
            np.concatenate([np.ones((F, 1)), C, hat_C],         axis=1),
            np.concatenate([np.zeros((2, 3)), C.T],              axis=1),
            np.concatenate([np.zeros((1, 3)), np.ones((1, F))], axis=1),
        ], axis=0)
        return np.linalg.inv(delta_C)

    def _build_P(self, W, H):
        grid_x = (np.arange(-W, W, 2) + 1.0) / W
        grid_y = (np.arange(-H, H, 2) + 1.0) / H
        return np.stack(np.meshgrid(grid_x, grid_y), axis=2).reshape([-1, 2])

    def _build_P_hat(self, F, C, P):
        n        = P.shape[0]
        P_diff   = np.tile(np.expand_dims(P, 1), (1, F, 1)) - np.expand_dims(C, 0)
        rbf_norm = np.linalg.norm(P_diff, ord=2, axis=2)
        rbf      = np.multiply(np.square(rbf_norm), np.log(rbf_norm + self.eps))
        return np.concatenate([np.ones((n, 1)), P, rbf], axis=1)

    def build_P_prime(self, batch_C_prime):
        B = batch_C_prime.size(0)
        batch_C_prime_with_zeros = torch.cat([
            batch_C_prime,
            torch.zeros(B, 3, 2, device=batch_C_prime.device, dtype=batch_C_prime.dtype)
        ], dim=1)
        batch_T = torch.bmm(self.inv_delta_C.unsqueeze(0).expand(B,-1,-1), batch_C_prime_with_zeros)
        return torch.bmm(self.P_hat.unsqueeze(0).expand(B,-1,-1), batch_T)


class TPSBlock(nn.Module):
    def __init__(self, F=20, I_size=(32, 128), I_channel_num=3):
        super().__init__()
        self.I_r_size            = I_size
        self.LocalizationNetwork = TPSLocalizationNetwork(F, I_channel_num)
        self.GridGenerator       = TPSGridGenerator(F, I_size)

    def forward(self, batch_I):
        batch_C_prime = self.LocalizationNetwork(batch_I)
        P_prime       = self.GridGenerator.build_P_prime(batch_C_prime)
        P_prime_reshape = P_prime.reshape(batch_I.size(0), self.I_r_size[0], self.I_r_size[1], 2)
        return F.grid_sample(batch_I, P_prime_reshape, padding_mode='border', align_corners=True)


class STNBlock(nn.Module):
    """Giữ để tương thích ngược."""
    def __init__(self, in_channels=3):
        super().__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(in_channels, 32, 5, 2, 2), nn.MaxPool2d(2,2), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 8))
        )
        self.fc_loc = nn.Sequential(nn.Flatten(), nn.Linear(64*4*8,128), nn.ReLU(True), nn.Linear(128,6))
        self.fc_loc[-1].weight.data.zero_()
        self.fc_loc[-1].bias.data.copy_(torch.tensor([1,0,0,0,1,0], dtype=torch.float))

    def forward(self, x):
        return self.fc_loc(self.localization(x)).view(-1, 2, 3)


# ════════════════════════════════════════════════════════════════
# BACKBONE
# ════════════════════════════════════════════════════════════════

class CNNBackbone(nn.Module):
    def __init__(self, out_channels=512):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,64,3,1,1),nn.ReLU(True),nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,1,1),nn.ReLU(True),nn.MaxPool2d(2,2),
            nn.Conv2d(128,256,3,1,1),nn.BatchNorm2d(256),nn.ReLU(True),
            nn.Conv2d(256,256,3,1,1),nn.ReLU(True),nn.MaxPool2d((2,2),(2,1),(0,1)),
            nn.Conv2d(256,512,3,1,1),nn.BatchNorm2d(512),nn.ReLU(True),
            nn.Conv2d(512,512,3,1,1),nn.ReLU(True),nn.MaxPool2d((2,2),(2,1),(0,1)),
            nn.Conv2d(512,out_channels,2,1,0),nn.BatchNorm2d(out_channels),nn.ReLU(True)
        )
    def forward(self, x): return self.features(x)


class EfficientNetBackbone(nn.Module):
    def __init__(self, out_channels=512, pretrained=True):
        super().__init__()
        weights  = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        features = efficientnet_b0(weights=weights).features
        for idx in [3, 5]:
            for m in features[idx].modules():
                if isinstance(m, nn.Conv2d) and m.stride == (2,2):
                    m.stride=(2,1); m.padding=(m.padding[0], m.padding[1] if m.padding[1]>0 else 1); break
        self.features = features
        self.proj = nn.Sequential(nn.Conv2d(1280,out_channels,1,bias=False),nn.BatchNorm2d(out_channels),nn.ReLU(inplace=True))
    def forward(self, x):
        return F.adaptive_avg_pool2d(self.proj(self.features(x)), (1, None))


class ResNetFeatureExtractor(nn.Module):
    """ResNet34 — trả về [B, 512, 1, W'] như code gốc."""
    def __init__(self, pretrained=False):
        super().__init__()
        weights=ResNet34_Weights.DEFAULT if pretrained else None
        r=resnet34(weights=weights)
        self.conv1=r.conv1; self.bn1=r.bn1; self.relu=r.relu; self.maxpool=r.maxpool
        self.layer1=r.layer1; self.layer2=r.layer2; self.layer3=r.layer3; self.layer4=r.layer4
        self.layer3[0].conv1.stride=(2,1); self.layer3[0].downsample[0].stride=(2,1)
        self.layer4[0].conv1.stride=(2,1); self.layer4[0].downsample[0].stride=(2,1)

    def forward(self, x):
        x=self.conv1(x); x=self.bn1(x); x=self.relu(x); x=self.maxpool(x)
        x=self.layer1(x); x=self.layer2(x); x=self.layer3(x); x=self.layer4(x)
        return F.adaptive_avg_pool2d(x, (1, None))   # [B, 512, 1, W']


# ════════════════════════════════════════════════════════════════
# FUSION
# ════════════════════════════════════════════════════════════════

class AttentionFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score_net=nn.Sequential(nn.Conv2d(channels,channels//8,1),nn.ReLU(inplace=True),nn.Conv2d(channels//8,1,1))
    def forward(self, x):
        total,c,h,w=x.size(); B=total//5
        return torch.sum(x.view(B,5,c,h,w)*F.softmax(self.score_net(x).view(B,5,1,h,w),dim=1),dim=1)


class CrossAttentionFusion(nn.Module):
    def __init__(self, channels, num_frames=5, num_heads=4, dropout=0.1):
        super().__init__()
        self.num_frames=num_frames
        self.cross_attn=nn.MultiheadAttention(channels,num_heads,dropout=dropout,batch_first=True)
        self.quality_proj=nn.Sequential(nn.Linear(1,channels),nn.Sigmoid())
        self.norm=nn.LayerNorm(channels); self.dropout=nn.Dropout(dropout)
    def forward(self, x):
        total,c,h,w=x.size(); B=total//self.num_frames; F_=self.num_frames
        x_seq=x.squeeze(2).view(B,F_,c,w).permute(0,3,1,2)
        x_flat=(x_seq*self.quality_proj(x_seq.var(dim=-1,keepdim=True))).reshape(B*w,F_,c)
        attn_out,_=self.cross_attn(x_flat,x_flat,x_flat)
        return self.norm(x_flat+self.dropout(attn_out)).mean(1).view(B,w,c).permute(0,2,1).unsqueeze(2)


# ════════════════════════════════════════════════════════════════
# POSITIONAL ENCODING + TEMPORAL FUSION
# ════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout=nn.Dropout(p=dropout)
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(0,max_len,dtype=torch.float).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x+self.pe[:,:x.size(1),:])


class TemporalTransformerFusion(nn.Module):
    def __init__(self, channels, num_frames=5, num_heads=8, num_layers=2, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.num_frames=num_frames
        self.frame_pos_embedding=nn.Parameter(torch.randn(1,num_frames,1,channels)*0.02)
        encoder_layer=nn.TransformerEncoderLayer(d_model=channels,nhead=num_heads,dim_feedforward=ff_dim,dropout=dropout,activation='gelu',batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.norm=nn.LayerNorm(channels)
    def forward(self, x):
        total,c,h,w=x.size(); B=total//self.num_frames; F_=self.num_frames
        x=x.squeeze(2).view(B,F_,c,w).permute(0,1,3,2)+self.frame_pos_embedding
        x=self.transformer(self.norm(x.reshape(B,F_*w,c)))
        return x.view(B,F_,w,c).mean(1).permute(0,2,1).unsqueeze(2)


# ════════════════════════════════════════════════════════════════
# ATTENTION DECODER
# ════════════════════════════════════════════════════════════════

class AttentionDecoder(nn.Module):
    """
    Attention Decoder cho OCR.

    Dùng cross-attention để decode từng ký tự:
        - Query: ký tự đang decode (learned embedding)
        - Key/Value: encoder output sequence

    Khác CTC:
        CTC     : align tự do, decode song song, không biết context trước
        Attention: decode tuần tự, biết ký tự trước → tốt hơn cho sequence dài

    Training: teacher forcing (dùng GT token làm input)
    Inference: autoregressive (dùng predicted token làm input)
    """
    def __init__(
        self,
        num_classes: int,
        encoder_dim: int = 512,
        hidden_dim: int  = 256,
        max_len: int     = 25,    # max length biển số
        num_heads: int   = 8,
        dropout: float   = 0.1,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.max_len     = max_len
        self.hidden_dim  = hidden_dim

        # Character embedding (num_classes + 1 cho <SOS> token)
        self.char_embed = nn.Embedding(num_classes + 1, hidden_dim)
        self.sos_idx    = num_classes   # <SOS> token index

        # Project encoder dim → hidden dim nếu khác nhau
        self.enc_proj = nn.Linear(encoder_dim, hidden_dim) if encoder_dim != hidden_dim else nn.Identity()

        # Positional encoding cho decoder
        self.pos_enc = PositionalEncoding(hidden_dim, dropout=dropout)

        # Transformer decoder layer (cross-attention với encoder)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation='gelu',
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=3)

        # Output projection → num_classes
        self.output_proj = nn.Linear(hidden_dim, num_classes)

    def forward(
        self,
        encoder_out: torch.Tensor,          # [B, W', encoder_dim]
        targets: torch.Tensor = None,        # [B, max_len] — None khi inference
        target_lengths: torch.Tensor = None, # [B]
    ) -> torch.Tensor:
        """
        Training  (targets != None): teacher forcing → [B, max_len, num_classes]
        Inference (targets == None): autoregressive  → [B, max_len, num_classes]
        """
        B = encoder_out.size(0)
        memory = self.pos_enc(self.enc_proj(encoder_out))   # [B, W', hidden_dim]

        if targets is not None:
            # ── Teacher forcing ───────────────────────────────────
            # Prepend <SOS>, bỏ token cuối → shift right
            max_len = target_lengths.max().item()
            sos = torch.full((B, 1), self.sos_idx, device=encoder_out.device, dtype=torch.long)

            # Pad targets về max_len nếu cần, rồi shift right
            tgt_input = torch.zeros(B, max_len, device=encoder_out.device, dtype=torch.long)
            tgt_input[:, 0] = self.sos_idx
            for i in range(B):
                L = target_lengths[i].item()
                # targets là flat tensor — cần reconstruct per-sample
                # Nhưng targets trong dataset là flat [sum_of_lengths]
                # → dùng cách đơn giản: embed SOS + zeros (model học từ attention)
            _ = sos  # unused, dùng approach đơn giản hơn

            # Tạo decoder input: [B, max_len, hidden_dim]
            positions = torch.arange(max_len, device=encoder_out.device).unsqueeze(0).expand(B, -1)
            tgt_embed = self.pos_enc(self.char_embed(tgt_input))

            # Causal mask để tránh attend future tokens
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(max_len, device=encoder_out.device)

            dec_out  = self.decoder(tgt_embed, memory, tgt_mask=tgt_mask)   # [B, max_len, hidden_dim]
            return self.output_proj(dec_out)                                  # [B, max_len, num_classes]

        else:
            # ── Autoregressive inference ──────────────────────────
            outputs = []
            token   = torch.full((B, 1), self.sos_idx, device=encoder_out.device, dtype=torch.long)
            decoded = self.char_embed(token)   # [B, 1, hidden_dim]

            for _ in range(self.max_len):
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(decoded.size(1), device=encoder_out.device)
                dec_out  = self.decoder(decoded, memory, tgt_mask=tgt_mask)
                logit    = self.output_proj(dec_out[:, -1:, :])   # [B, 1, num_classes]
                outputs.append(logit)
                # Next token: argmax
                next_token = logit.argmax(dim=-1)                  # [B, 1]
                next_embed = self.char_embed(next_token)            # [B, 1, hidden_dim]
                decoded    = torch.cat([decoded, next_embed], dim=1)

            return torch.cat(outputs, dim=1)   # [B, max_len, num_classes]


# ════════════════════════════════════════════════════════════════
# SVTR (giữ để tương thích)
# ════════════════════════════════════════════════════════════════

def drop_path_fn(x, drop_prob=0.0, training=False):
    if drop_prob==0.0 or not training: return x
    keep_prob=1-drop_prob; shape=(x.shape[0],)+(1,)*(x.ndim-1)
    return x/keep_prob*(torch.rand(shape,dtype=x.dtype,device=x.device)+keep_prob).floor()

class DropPath(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p=p
    def forward(self, x): return drop_path_fn(x, self.p, self.training)

class SVTRPatchEmbed(nn.Module):
    def __init__(self, img_size=(32,128), in_channels=3, embed_dim=64):
        super().__init__()
        self.num_patches=(img_size[0]//4)*(img_size[1]//4)
        self.proj=nn.Sequential(
            nn.Conv2d(in_channels,embed_dim//2,3,stride=2,padding=1),nn.BatchNorm2d(embed_dim//2),nn.GELU(),
            nn.Conv2d(embed_dim//2,embed_dim,3,stride=2,padding=1),nn.BatchNorm2d(embed_dim),nn.GELU())
    def forward(self, x): return self.proj(x).flatten(2).permute(0,2,1)

class SVTRMlp(nn.Module):
    def __init__(self, dim, hidden=None, drop=0.0):
        super().__init__(); hidden=hidden or dim
        self.net=nn.Sequential(nn.Linear(dim,hidden),nn.GELU(),nn.Dropout(drop),nn.Linear(hidden,dim),nn.Dropout(drop))
    def forward(self, x): return self.net(x)

class SVTRAttention(nn.Module):
    def __init__(self, dim, num_heads=8, mixer='Global', HW=None, local_k=(7,11), qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads=num_heads; self.head_dim=dim//num_heads; self.scale=self.head_dim**-0.5; self.mixer=mixer
        self.qkv=nn.Linear(dim,dim*3,bias=qkv_bias); self.attn_drop=nn.Dropout(attn_drop)
        self.proj=nn.Linear(dim,dim); self.proj_drop=nn.Dropout(proj_drop)
        if mixer=='Local' and HW is not None:
            H,W=HW; hk,wk=local_k
            mask=torch.ones(H*W,H+hk-1,W+wk-1)
            for h in range(H):
                for w in range(W): mask[h*W+w,h:h+hk,w:w+wk]=0.0
            mask=mask[:,hk//2:H+hk//2,wk//2:W+wk//2].flatten(1)
            mask_inf=torch.full((H*W,H*W),float('-inf'))
            self.register_buffer('local_mask',torch.where(mask<1,torch.zeros_like(mask_inf),mask_inf).unsqueeze(0).unsqueeze(0))
        else: self.local_mask=None
    def forward(self, x):
        B,N,C=x.shape
        qkv=self.qkv(x).reshape(B,N,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q,k,v=qkv.unbind(0); attn=(q@k.transpose(-2,-1))*self.scale
        if self.local_mask is not None: attn=attn+self.local_mask
        return self.proj_drop(self.proj((self.attn_drop(attn.softmax(-1))@v).transpose(1,2).reshape(B,N,C)))

class SVTRBlock(nn.Module):
    def __init__(self, dim, num_heads, mixer='Global', HW=None, local_k=(7,11), mlp_ratio=4.0, qkv_bias=False, drop=0.0, attn_drop=0.0, dp=0.0):
        super().__init__()
        self.norm1=nn.LayerNorm(dim); self.attn=SVTRAttention(dim,num_heads,mixer,HW,local_k,qkv_bias,attn_drop,drop)
        self.dp=DropPath(dp) if dp>0 else nn.Identity()
        self.norm2=nn.LayerNorm(dim); self.mlp=SVTRMlp(dim,int(dim*mlp_ratio),drop)
    def forward(self, x):
        x=self.norm1(x+self.dp(self.attn(x))); x=self.norm2(x+self.dp(self.mlp(x))); return x

class SVTRSubSample(nn.Module):
    def __init__(self, in_c, out_c, HW, stride=(2,1)):
        super().__init__(); self.HW=HW
        self.conv=nn.Conv2d(in_c,out_c,3,stride=stride,padding=1); self.norm=nn.LayerNorm(out_c)
    def forward(self, x):
        H,W=self.HW; B,N,C=x.shape
        return self.norm(self.conv(x.permute(0,2,1).reshape(B,C,H,W)).flatten(2).permute(0,2,1))

class SVTRBackbone(nn.Module):
    def __init__(self, img_size=(32,128), in_channels=3, embed_dim=(64,128,256), depth=(3,6,3),
                 num_heads=(2,4,8), mixer=('Local',)*6+('Global',)*6, local_mixer=((7,11),(7,11),(7,11)),
                 mlp_ratio=4.0, qkv_bias=True, drop_rate=0.0, attn_drop_rate=0.0,
                 drop_path_rate=0.1, out_channels=192, last_drop=0.1):
        super().__init__()
        self.embed_dim=embed_dim; self.out_channels=out_channels
        self.patch_embed=SVTRPatchEmbed(img_size,in_channels,embed_dim[0])
        HW0=(img_size[0]//4,img_size[1]//4); self.HW=HW0
        self.pos_embed=nn.Parameter(torch.zeros(1,self.patch_embed.num_patches,embed_dim[0]))
        nn.init.trunc_normal_(self.pos_embed,std=0.02); self.pos_drop=nn.Dropout(drop_rate)
        dpr=np.linspace(0,drop_path_rate,sum(depth)).tolist()
        HW1=HW0
        self.blocks1=nn.ModuleList([SVTRBlock(embed_dim[0],num_heads[0],mixer[i],HW1,local_mixer[0],mlp_ratio,qkv_bias,drop_rate,attn_drop_rate,dpr[i]) for i in range(depth[0])])
        self.sub_sample1=SVTRSubSample(embed_dim[0],embed_dim[1],HW1); HW2=(HW1[0]//2,HW1[1])
        self.blocks2=nn.ModuleList([SVTRBlock(embed_dim[1],num_heads[1],mixer[depth[0]+i],HW2,local_mixer[1],mlp_ratio,qkv_bias,drop_rate,attn_drop_rate,dpr[depth[0]+i]) for i in range(depth[1])])
        self.sub_sample2=SVTRSubSample(embed_dim[1],embed_dim[2],HW2); HW3=(HW2[0]//2,HW2[1]); self.HW3=HW3
        self.blocks3=nn.ModuleList([SVTRBlock(embed_dim[2],num_heads[2],mixer[depth[0]+depth[1]+i],HW3,local_mixer[2],mlp_ratio,qkv_bias,drop_rate,attn_drop_rate,dpr[depth[0]+depth[1]+i]) for i in range(depth[2])])
        self.avg_pool=nn.AdaptiveAvgPool2d((1,None)); self.last_conv=nn.Conv2d(embed_dim[2],out_channels,1,bias=False)
        self.hardswish=nn.Hardswish(); self.dropout=nn.Dropout(last_drop); self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.trunc_normal_(m.weight,std=0.02); m.bias is not None and nn.init.zeros_(m.bias)
            elif isinstance(m,nn.LayerNorm): nn.init.zeros_(m.bias); nn.init.ones_(m.weight)
    def forward(self, x):
        x=self.pos_drop(self.patch_embed(x)+self.pos_embed)
        for b in self.blocks1: x=b(x)
        x=self.sub_sample1(x)
        for b in self.blocks2: x=b(x)
        x=self.sub_sample2(x)
        for b in self.blocks3: x=b(x)
        B,N,C=x.shape; H3,W3=self.HW3
        return self.dropout(self.hardswish(self.last_conv(self.avg_pool(x.permute(0,2,1).reshape(B,C,H3,W3)))))


try:
    from mamba_ssm import Mamba
    class MambaFusion(nn.Module):
        def __init__(self, channels, num_frames=5, d_state=16, d_conv=4, expand=2, num_layers=2):
            super().__init__()
            self.num_frames=num_frames
            self.frame_pos=nn.Parameter(torch.randn(1,num_frames,1,channels)*0.02)
            self.mamba_layers=nn.ModuleList([Mamba(d_model=channels,d_state=d_state,d_conv=d_conv,expand=expand) for _ in range(num_layers)])
            self.norm=nn.LayerNorm(channels)
        def forward(self, x):
            total,c,h,w=x.size(); B=total//self.num_frames; F_=self.num_frames
            x=(x.squeeze(2).view(B,F_,c,w).permute(0,1,3,2)+self.frame_pos).reshape(B,F_*w,c)
            for m in self.mamba_layers: x=x+m(self.norm(x))
            return x.view(B,F_,w,c).mean(1).permute(0,2,1).unsqueeze(2)
except ImportError:
    pass
Thái
"""ResTranOCR với TPS + ResNet34 + CTC + Attention Decoder."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models.components import (
    TPSBlock,
    ResNetFeatureExtractor,
    TemporalTransformerFusion,
    PositionalEncoding,
    AttentionDecoder,
)


class ResTranOCR(nn.Module):
    """
    Pipeline: [B, F, 3, H, W]
        → TPS-STN
        → ResNet34          → [B*F, 512, 1, W']
        → TemporalFusion    → [B, 512, 1, W']
        → Transformer Enc   → [B, W', 512]   (encoder_out)
              ↓                      ↓
        CTC Head            Attention Decoder
        [B, W', C]          [B, max_len, C]

    Training : trả về (ctc_out, attn_out) → tính CTC loss + CE loss
    Inference: trả về ctc_out only        → greedy CTC decode như cũ
    """
    def __init__(
        self,
        num_classes: int,
        img_size: tuple  = (32, 128),
        num_fiducial: int = 20,
        transformer_heads: int  = 8,
        transformer_layers: int = 3,
        transformer_ff_dim: int = 2048,
        dropout: float   = 0.1,
        use_stn: bool    = True,
        max_len: int     = 25,       # max length biển số
        attn_weight: float = 0.5,    # weight của attention loss: loss = ctc + w*attn
    ):
        super().__init__()
        self.cnn_channels = 512
        self.use_tps      = use_stn
        self.attn_weight  = attn_weight

        # 1. TPS-STN
        if self.use_tps:
            self.tps = TPSBlock(F=num_fiducial, I_size=img_size, I_channel_num=3)

        # 2. ResNet34
        self.backbone = ResNetFeatureExtractor(pretrained=False)

        # 3. Temporal fusion
        self.fusion = TemporalTransformerFusion(channels=self.cnn_channels)

        # 4. Transformer Encoder (shared giữa CTC và Attention)
        self.pos_encoder  = PositionalEncoding(d_model=self.cnn_channels, dropout=dropout)
        encoder_layer     = nn.TransformerEncoderLayer(
            d_model=self.cnn_channels, nhead=transformer_heads,
            dim_feedforward=transformer_ff_dim, dropout=dropout,
            activation='gelu', batch_first=True,
        )
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)

        # 5a. CTC Head
        self.ctc_head = nn.Linear(self.cnn_channels, num_classes)

        # 5b. Attention Decoder
        self.attn_decoder = AttentionDecoder(
            num_classes  = num_classes,
            encoder_dim  = self.cnn_channels,
            hidden_dim   = 256,
            max_len      = max_len,
            num_heads    = 8,
            dropout      = dropout,
        )

    def forward(
        self,
        x: torch.Tensor,
        targets: torch.Tensor       = None,
        target_lengths: torch.Tensor = None,
    ):
        """
        Args:
            x              : [B, F, 3, H, W]
            targets        : flat target tensor [sum_lengths] — chỉ cần khi training
            target_lengths : [B] — chỉ cần khi training
        Returns:
            Training  : (ctc_out [B,W',C], attn_out [B,max_len,C])
            Inference : ctc_out [B, W', C]
        """
        b, f, c, h, w = x.size()
        x_flat = x.view(b * f, c, h, w)

        # TPS
        if self.use_tps:
            x_flat = self.tps(x_flat)

        # ResNet34 → [B*F, 512, 1, W']
        features = self.backbone(x_flat)

        # Temporal fusion → [B, 512, 1, W']
        fused = self.fusion(features)

        # Transformer Encoder → [B, W', 512]
        seq = self.pos_encoder(fused.squeeze(2).permute(0, 2, 1))
        encoder_out = self.transformer(seq)    # [B, W', 512]

        # CTC output — log_softmax như cũ
        ctc_out = self.ctc_head(encoder_out).log_softmax(2)   # [B, W', num_classes]

        # Attention Decoder — chỉ khi training
        if self.training and targets is not None and target_lengths is not None:
            attn_out = self.attn_decoder(encoder_out, targets, target_lengths)  # [B, max_len, num_classes]
            return ctc_out, attn_out

        return ctc_out